#كيف يمكن التنبؤ بالأحداث الكارثية؟



## مقدمة 



في نهاية هذا البرنامج التعليمي القصير ولكن الأكثر دقة، ستفهم ما هو GEV (توزيع القيمة القصوى المعمم) وكيفية ملاءمته مع بياناتك في لغة بايثون. 
ما نحتاج إلى التفكير فيه قبل التحدث عن إحصاءات القيم المتطرفة، هو أن الإحصاءات العامة تركز على القيم المتوسطة والظواهر التي يمكن ملاحظتها، والتي **لدينا بيانات** لها. تختلف إحصائيات القيم المتطرفة عن الإحصائيات "المتوسطة" في هذا الصدد، وفي الواقع، كما سنرى، نحن هنا نحاول وضع نموذج لشيء **ليس لدينا بيانات عنه!** وسوف ترى قريبًا ما نعنيه بذلك. قبل أن أصل إلى هناك، أريد أن أوضح لك لماذا تعتبر نمذجة القيم المتطرفة مشكلة فعلية، ذات أهمية كبيرة لشركات التأمين والخدمات العامة. لنبدأ بمثالين.



### <center> مونبلييه، فرنسا، 2014
**3 ساعات من المطر = 50% من إجمالي المتوسط السنوي.** *من غير المعروف أن مونبلييه هي مدينة ذات مستوى أمطار سنوي مرتفع، وتقع في موقع جاف في بروفانس، لذلك إذا نظرنا إلى متوسط القيمة فلدينا قيمة شائعة جدًا. ولكن لا يستبعد أنه في بعض الأحيان، نادراً، تظهر قيم عالية جداً، كما نرى:*
<img align="center" src="@@KEEP_00000@@ style="width: 700px;"/>
<br>



### <center> نهر سالتينا في بريغ، سويسرا، 1993
** التدفقات والأمطار المقاسة غير المسبوقة في القرن العشرين **.
*نفس المشكلة هنا، هذا النهر لديه تدفقات معقولة في المتوسط، ولكن في بعض الأحيان يمكن أن يكون التدفق استثنائيًا.*
<img align="center" src="@@KEEP_00001@@ style="width: 600px;"/>
<br>



### <center> عدد الكوارث الطبيعية منذ عام 1980 في تزايد 
**تميل تلك الأحداث "المتطرفة" إلى التزايد في تلك الأيام، وتجعل تحليل القيمة المتطرفة "شائعًا".**
<img align="center" src="@@KEEP_00002@@ style="width: 600px;"/>
<br>


### <center> قد يتساءل البعض عن النموذج الإحصائي الذي يجب أن نستخدمه للتنبؤ بخسارة نادال في رولان جاروس 
<img align="center" src="@@KEEP_00003@@ style="width: 600px;"/>
<br>
<center> (تقريبًا) مجرد مزاح



## سؤال: ماذا لو أردنا قياس احتمالية وقوع مثل هذه الأحداث؟



كما قلنا، فإن خصوصية تحليل القيمة القصوى لا تتمثل في وضع نموذج لظاهرة متوسطة بل لظاهرة متطرفة.



من أجل فهم ما نعنيه بذلك، دعونا نعمل مع مجموعة بيانات نهر Nidd هنا: https://www.stat.auckland.ac.nz/~wild/data/Rdatasets/csv/evir/nidd.thresh.csv, فهي تحتوي على $n=154$ قياسات ثلاثية لتدفقات النهر (في $m³/s$)، على مدى 38.5 عامًا (المصدر: أبحاث البيئة الطبيعية، 1975). 
لنرسم رسمًا بيانيًا للبيانات.


In [ ]:
import numpy as np
import pandas as pd

%matplotlib inline
import matplotlib.pyplot as plt

df = pd.read_csv("data/nidd.thresh.csv")
df = df["x"]
df.hist()
plt.xlabel("Flow in m³/s")
plt.ylabel("Number of measures")
plt.title("Histogram of Nidd river's flows, mean=%d m³/s" % df.mean());


نرى أن معظم التدفقات تقع تحت $80$ $m³/s$، وجزء كبير منها يقع تحت $100$ $m³/s$، لكننا نرى أيضًا أنه حتى لو كانت التدفقات منخفضة بشكل عام، فقد تصبح أيضًا $300$ $m³/s$ عالية.
عادةً ما تكون هذه مشكلة حيث لا يمكننا استخدام القيمة المتوسطة لتمثيل أي شيء، نظرًا لأن معظم التدفقات تكون ضمن $80$ $m³/s$، وبعضها مرتفع جدًا.



**أسئلة تمهيدية: ما هو احتمال ملاحظة تدفق أعلى من $160$ $m³/s$؟ ثم $255$ $m³/s$ ؟**


In [ ]:
n, bins, patches = plt.hist(df)
for i, x in enumerate((bins > 160)):
    if x and i < len(bins) - 1:
        patches[i].set_fc("r")

plt.show()


تمثل خطوة السلة الأخيرة تدفقًا واحدًا، وبهذه الطريقة يمكننا حتى حساب عدد السكان في جميع الصناديق الحمراء بصريًا:
$$P(X > 160) \simeq nb(X_{i} > 160) / n = 11/154 = 1/14$$
باستخدام بياناتنا التجريبية، نحسب بهذه الطريقة احتمالية وجود تدفق أعلى من $160$ $ m³/s$، مما يخبرنا أن لدينا تدفق من هذا النوع في المتوسط كل $3.5$ سنوات.
لا شيء غير عادي حتى الآن، أليس كذلك؟


In [ ]:
n, bins, patches = plt.hist(df)
for i, x in enumerate((bins > 255)):
    if x and i < len(bins) - 1:
        patches[i].set_fc("r")

plt.show()


الرسم البياني:
$$P(X > 255) \simeq nb(X_{i} > 255) / n = 3/154 $$هنا مرة أخرى يمكننا الإجابة (بصعوبة) بأن لدينا تدفقًا يتفوق على $255$ $m³/s$ في المتوسط ​​كل $12$،$8$ سنوات.
حسنًا، كفى لعبًا، فلنصل إلى السؤال المزعج:



** سؤال "ليس لديك طريقة للإجابة": الآن، ما هو احتمال ملاحظة مثل هذا التدفق المتفوق على $400$ $m³/s$؟**


In [ ]:
n, bins, patches = plt.hist(df)
for i, x in enumerate((bins > 400)):
    if x and i < len(bins) - 1:
        patches[i].set_fc("r")

plt.show()


الرسم البياني:
$$P(X > 400) \simeq nb(X_{i} > 400) / n = 0 $$
الإجابة ليست مرضية، لأنه بالتأكيد ليس "من المستحيل" ملاحظة تدفق أعلى من $400$ $m³/s$.
## استنتاج جزئي: لا يمكننا تقدير احتمالية مثل هذا التدفق باستخدام الرسم البياني!
والآن وصلنا بدقة إلى ما تحدثنا عنه من قبل، نريد حساب احتمالية حدث ليس لدينا أي بيانات عنه على الإطلاق، وآمل أن يكون هذا واضحًا الآن. في النمذجة الإحصائية، عادةً ما نقوم بتدريب خوارزمية باستخدام مجموعة بيانات من المفترض أن يكون لها نفس توزيع بيانات الاختبار، وبعبارة أخرى، لدينا بيانات لما نريد تصميمه. تلعب نظرية القيمة القصوى دورًا هنا:
- **لا تسمح الإحصائيات "الكلاسيكية" بحساب الاحتمالات مثل $P(X > x)$ عندما يكون $x$ يتجاوز الحد الأقصى لملاحظاتنا.**
- **نأمل أن تمنحنا نظرية القيمة المتطرفة أدوات لاستقراء ما هو أبعد من بياناتنا.**
نكتة "مشهورة" توضح هذه الإشكالية: أثناء الليل، يجلس شخص القرفصاء عند أسفل عمود إنارة، فيسأله أحد المارة: "ماذا تفعل؟" "أنا أبحث عن مفاتيحي." "هل أنت متأكد أنك فقدتهم حول عمود الإنارة؟"، يسأل المارة مرة أخرى. "لا، ولكن، في الواقع، هذا هو المكان الوحيد المضاء."



إحصائيات القيمة القصوى تشبه هذه النكتة نوعًا ما، نريد إجراء إحصائيات حيث لا يوجد ضوء، حيث لا توجد بيانات.


## النظرية: نظرية القيم المتطرفة / التنفيذ: scipy.stats.genextreme



يمكن اعتبار نظرية القيم المتطرفة بمثابة نظير لنظرية الحد المركزي.
بالنسبة للمتغير العشوائي $X$ والحد الأقصى الذي يزيد عن n من الملاحظات: $X_{n,n}$. 
في ظل الظروف العامة لتوزيع X، هناك ثلاث معلمات
$a_{n}$ و$b_{n}$ و$\gamma$ بحيث:
   $$\lim_{n -> +\infty} P(\frac{X_{n,n} - a_{n}}{b_{n}} \leq x) = H_{\gamma}(x)$$
   حيث:
$$ H_{\gamma}(x) = \begin{cases}
                \exp(-\exp(-x))            &\text{for } \gamma = 0\\
                \exp(-(1+\gamma x)^{-1/\gamma}_{+})   &\text{for }
                                                        \gamma != 0
              \end{cases}
$$
   
   مع $y_{+} = max(0,y)$.
   
مفردات مفيدة:
 - $H_{\gamma}$ هو **توزيع القيمة القصوى المعممة (GEV)**.
 - $\gamma$ هي معلمة الشكل
 - $a_{n}$ و$b_{n}$ عبارة عن معلمات قياس.
 
مع قيم مختلفة لـ $\gamma$، يوجد النوع الأول والنوع الثاني والنوع الثالث GEV، ويسمى أيضًا توزيعات Gumbel ($\gamma=0$)، وFréchet ($\gamma >0$) وWeibull ($\gamma <0$).
إذا كان لديك متغير عشوائي $X$ وقمت بدراسة توزيع الحد الأقصى له على عدد من العينات $X_{n,n}$، فيجب أن ينتمي هذا التوزيع ويجب أن "يقع" في مجال جذب أحد هذه الأنواع الثلاثة من GEV. 
 
(المزيد هنا: https://en.wikipedia.org/wiki/Generalized_extreme_value_distribution.)
** سوف نستخدم فئة genextreme من scipy **


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import genextreme

plt.figure(figsize=(10, 8))
x = np.linspace(-4, 4, 201)
# Please notice that in genextreme the shape parameter (gamma) equals to -gamma in our equations here
plt.plot(
    x,
    genextreme.pdf(x, -0.6),
    color="#00FF00",
    label='Fréchet (\u03B3=0.6) : "Heavy tails" distributions\
 (Cauchy, Student, Pareto..)',
)
plt.plot(
    x,
    genextreme.pdf(x, 0.0),
    "r",
    label='Gumbel (\u03B3=0): "Light tails" distributions\
 (Gaussian, Log-normal, Gamma, Exponential)',
)
plt.plot(
    x,
    genextreme.pdf(x, 0.6),
    "b",
    label='Weibull (\u03B3=-0.6): "Finite tails" distributions\
 (Uniform, Beta)',
)
plt.ylim(-0.01, 0.51)
plt.legend(loc="upper left")
plt.xlabel("x")
plt.ylabel("Density")
plt.title("Generalized Extreme Value Distributions")
plt.show()


دعونا نستخدم مرة أخرى مجموعة البيانات الخاصة بنهر Nidd، وسوف نلائم GEV عليها ونقدر الكميات المتطرفة.
من الناحية العملية، سنقوم بتقدير المعلمات الثلاثة $\gamma$ و$a_{n}$ و$b_{n}$ بحيث:
$$P(X_{n,n} \leq x) \simeq H_{\gamma}(\frac{x-a_{n}}{b_{n}})$$
لذا نحتاج أولاً إلى تحويل بياناتنا الثلاثية، وهي ملاحظاتنا $X$ إلى ملاحظات X_{n,n}، لنفترض الحد الأقصى السنوي، وسننشئ متجهًا للملاحظات عن طريق أخذ الحد الأقصى للتدفق لكل عام:


In [ ]:
for i in range(0, int(df.shape[0] / 4)):
    if i == 0:
        df_max = df.iloc[4 * i : 4 * i + 4].max()
    else:
        df_max = np.append(df_max, df.iloc[4 * i : 4 * i + 4].max())

In [ ]:
plt.hist(df_max)

In [ ]:
shape, loc, scale = genextreme.fit(df_max)

In [ ]:
-shape

تخبرنا معلمات الشكل هذه أن GEV هو من نوع Fréchet، مع ذيول ثقيلة، مما يعني أن تدفق نهرنا يمكن أن يكون كبيرًا جدًا في بعض الأحيان، ومن الأفضل أن نكون حذرين وأن نبني حماية قوية من الفيضانات.



يمكننا رسم التوزيع المناسب لنرى أنه منطقي


In [ ]:
plt.hist(df_max, normed=True)

x = np.linspace(df_max.min(), df_max.max(), 1000)
y = genextreme.pdf(x, shape, loc, scale)
plt.plot(x, y, "c", linewidth=3);


**يمكننا الآن حساب احتمال أن يتجاوز تدفق النهر 350 $m^3/s$:**
في الواقع، لأن ملاحظات $X$ التي لدينا مستقلة، وبالتالي:
$P(X_{n,n} \leq x) = F^{n}(x)$
ومن نظرية القيمة القصوى يعني هذا $F(x) \simeq H_{\gamma}^{1/n}(\frac{x-a_{n}}{b_{n}})$، فأخذ السجل وتطويره حول $\log(1+u)$ يعطي:
$P(X\geq x) \simeq -\frac{1}{n}\log H_{\gamma}(\frac{x-a_{n}}{b_{n}})$
لذلك لدينا الآن تقريب لوظيفة البقاء في الذيل:
$$ \hat{F}(x) = P(X\geq x) \simeq \begin{cases}
                \frac{1}{n}[1+\gamma(\frac{x-a_{n}}{b_{n}})]^{-1/\gamma}        &\text{for } \gamma != 0\\
                \frac{1}{n}\exp(-(\frac{x-a_{n}}{b_{n}}))   &\text{for }
                                                        \gamma = 0
              \end{cases}
$$


In [ ]:
def surv(x, n, shape, loc, scale):
    if shape != 0:
        surv = 1 / n * (1 - shape * (x - loc) / scale) ** (1 / shape)
    else:
        surv = 1 / n * np.exp(-(x - loc) / scale)
    return surv

In [ ]:
surv(400, 4, shape, loc, scale)

In [ ]:
1 / 0.004 / 4


لذا فإن احتمال أن يكون التدفق الثلثي أكبر من $400$ $m³/s$ هو حوالي 0.004، مما يعني ثلثًا واحدًا في 250، أي مرة واحدة في 62.5 سنة! *بدلاً من "أبداً" من قبل*
يمكننا أيضًا الإجابة على أسئلة مثل: ما هو تدفق النهر الذي نتوقع أن نتجاوزه كل قرن، لأن لدينا أيضًا معكوس دالة البقاء:
$$ \hat{F}^{-1}(x)\simeq \begin{cases}
                a_{n} + \frac{b_{n}}{\gamma}[(np)^{-\gamma} - 1]         &\text{for } \gamma != 0\\
                a_{n} - b_{n}\log(np)   &\text{for }
                                                        \gamma = 0
              \end{cases}
$$


In [ ]:
def invsurv(p, n, shape, loc, scale):
    if shape != 0:
        invsurv = loc + scale / -shape * ((n * p) ** shape - 1)
    else:
        invsurv = loc - scale * np.log(n * p)
    return invsurv

In [ ]:
invsurv(1 / 400, 4, shape, loc, scale)


قمنا هنا بحساب التدفق الذي سنحصل عليه كل 400 فصل، أي كل قرن: $452$ $m³/s$!
من المؤسف جدًا أن هؤلاء الأشخاص لم يتبعوا البرنامج التعليمي الخاص بي من قبل: <center> 
<img align="center" src="@@KEEP_00006@@ style="width: 600px;"/>
<br>



#الخلاصة


**ما هو الجديد :**
- أنت الآن تفهم مشكلة الكميات المتطرفة في الإحصاء الكلاسيكي.
- أنت تعلم الآن أن هناك نظرية القيمة القصوى التي تشبه CLT والتي تخبرنا بوجود 3 أنواع من التوزيعات للحد الأقصى لأي متغير عشوائي، اعتمادًا على معلمة الشكل $\gamma$: Gumbel، Fréchet، Weibull.
- لملاءمة هذا التوزيع، نحتاج إلى مجموعة بيانات بحد أقصى، ولتقدير $\gamma$ و$a_{n}$ (loc) و$b_{n}$ (مقياس).
- يمكننا القيام بذلك باستخدام scipy.stats.genextreme بسرعة كبيرة.
- من خلاله يمكننا حساب الكميات المتطرفة واحتمالات القيمة المتطرفة حيث ليس لدينا بيانات لنتعلمها !!!
**ما ليس جديدًا بعد:**
- من الصعب في بعض الأحيان تكوين مجموعة بيانات من الحدود القصوى، وهناك أيضًا بديل مماثل بمستويات العتبة بدلاً من الحدود القصوى، وهو أسهل بشكل عام. والتوزيع المناسب يسمى GPD (توزيع باريتو المعمم)، وهو موجود في scipy.stats.pareto.



**شكرًا لاهتمامكم!** **بارك الله فيكم!**